Model

In [ ]:
import numpy as np

class DecisionTreeNode:
    def __init__(self, feature=None, threshold=None, left=None, right=None, label=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.label = label


class CustomDecisionTree:
    def __init__(self, max_depth=10, min_samples=5, n_features=None):
        self.max_depth = max_depth
        self.min_samples = min_samples
        self.n_features = n_features
        self.root = None

    def fit(self, X, y):
        self.n_classes = len(np.unique(y))
        self.n_features_total = X.shape[1]
        self.n_features = self.n_features or self.n_features_total
        self.root = self._build_tree(X, y, depth=0)

    def _build_tree(self, X, y, depth):
        n_samples = X.shape[0]

        # stopping conditions
        if (
            depth >= self.max_depth or
            n_samples < self.min_samples or
            len(np.unique(y)) == 1
        ):
            return DecisionTreeNode(label=self._majority_label(y))

        feature_idxs = np.random.choice(self.n_features_total, self.n_features, replace=False)

        best_feature, best_threshold = self._best_split(X, y, feature_idxs)

        if best_feature is None:
            return DecisionTreeNode(label=self._majority_label(y))

        left_mask = X[:, best_feature] < best_threshold
        right_mask = ~left_mask

        left = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right = self._build_tree(X[right_mask], y[right_mask], depth + 1)

        return DecisionTreeNode(best_feature, best_threshold, left, right)

    def _best_split(self, X, y, feature_idxs):
        best_gini = float("inf")
        best_feature, best_threshold = None, None

        for feature in feature_idxs:
            X_col = X[:, feature]
            thresholds = np.unique(X_col)

            step = max(1, len(thresholds) // 10)

            for t in thresholds[::step]:
                gini = self._gini_split(y, X_col, t)

                if gini < best_gini:
                    best_gini = gini
                    best_feature = feature
                    best_threshold = t

        return best_feature, best_threshold

    def _gini_split(self, y, X_col, threshold):
        left = y[X_col < threshold]
        right = y[X_col >= threshold]

        if len(left) == 0 or len(right) == 0:
            return float("inf")

        def gini(group):
            if len(group) == 0:
                return 0

            classes, counts = np.unique(group, return_counts=True)

            probs = counts / np.sum(counts)
            return 1 - np.sum(probs ** 2)

        n = len(y)
        return (len(left)/n)*gini(left) + (len(right)/n)*gini(right)

    def _majority_label(self, y):
        classes, counts = np.unique(y, return_counts=True)
        return classes[np.argmax(counts)]

    def predict(self, X):
        return np.array([self._traverse(x, self.root) for x in X])

    def _traverse(self, x, node):
        if node.label is not None:
            return node.label

        if x[node.feature] < node.threshold:
            return self._traverse(x, node.left)
        else:
            return self._traverse(x, node.right)

Training and Testing

In [19]:
import importlib
import preprocessing2
importlib.reload(preprocessing2)

<module 'preprocessing2' from 'c:\\~~~GAM3A~~~\\Semester 6\\machine learning\\project\\Image-classification-ML-Project\\phase_2\\preprocessing2.py'>

Cross Validation

In [ ]:
from preprocessing2 import preprocess, custom_classification_report, custom_confusion_matrix, custom_accuracy_score, k_fold_indices, custom_macro_f1_score
import numpy as np

feature_methods = ["cnn", "hog", "pca","flatten"]


best_f1_score = 0
best_params = {}

current_run = 1

for feature_method in feature_methods:
    X_train, y_train, X_val, y_val, X_test, y_test, _ = preprocess(
        feature_method=feature_method,
        n_pca=50
    )

    folds = k_fold_indices(X_train, k=3)

    param_grid = {
    'max_depth': [8, 10, 12],
    'min_samples': [5, 10],
    'n_features': [int(np.sqrt(X_train.shape[1])), X_train.shape[1] // 2, X_train.shape[1]]
    }
    total_runs = len(feature_methods) * len(param_grid['max_depth']) * len(param_grid['min_samples']) * len(param_grid['n_features'])

    for depth in param_grid['max_depth']:
        for min_s in param_grid['min_samples']:
            for n_feat in param_grid['n_features']:

                print(f"--- Run {current_run}/{total_runs} | Feature:{feature_method} | Depth:{depth} | MinSamples:{min_s} | n_feat:{n_feat} ---")

                fold_f1_scores = []

                for train_idx, val_idx in folds:
                    X_fold_train, y_fold_train = X_train[train_idx], y_train[train_idx]
                    X_fold_val, y_fold_val = X_train[val_idx], y_train[val_idx]

                    cv_model = CustomDecisionTree(
                        max_depth=depth,
                        min_samples=min_s,
                        n_features=min(n_feat, X_train.shape[1]),
                    )

                    cv_model.fit(X_fold_train, y_fold_train)
                    preds = cv_model.predict(X_fold_val)

                    fold_f1 = custom_macro_f1_score(y_fold_val, preds, n_classes=10)
                    fold_f1_scores.append(fold_f1)

                avg_f1 = np.mean(fold_f1_scores)
                print(f"    -> 3-Fold Average Macro F1: {avg_f1:.4f}\n")

                if avg_f1 > best_f1_score:
                    best_f1_score = avg_f1
                    best_params = {
                        'feature_method': feature_method,
                        'max_depth': depth,
                        'min_samples': min_s,
                        'n_features': n_feat
                    }

                current_run += 1

print("="*50)
print("  GRID SEARCH COMPLETE")
print("="*50)
print(f"Best CV Macro F1: {best_f1_score:.4f}")
print(f"Best Parameters: {best_params}")

Extracting CNN Features...
141/141 ━━━━━━━━━━━━━━━━━━━━ 6s 39ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 12s 37ms/step
--- Run 1/72 | Feature:cnn | Depth:8 | MinSamples:5 | n_feat:22 ---
    -> 3-Fold Average Macro F1: 0.5624

--- Run 2/72 | Feature:cnn | Depth:8 | MinSamples:5 | n_feat:256 ---
    -> 3-Fold Average Macro F1: 0.6500

--- Run 3/72 | Feature:cnn | Depth:8 | MinSamples:5 | n_feat:512 ---
    -> 3-Fold Average Macro F1: 0.6504

--- Run 4/72 | Feature:cnn | Depth:8 | MinSamples:10 | n_feat:22 ---
    -> 3-Fold Average Macro F1: 0.5629

--- Run 5/72 | Feature:cnn | Depth:8 | MinSamples:10 | n_feat:256 ---
    -> 3-Fold Average Macro F1: 0.6501

--- Run 6/72 | Feature:cnn | Depth:8 | MinSamples:10 | n_feat:512 ---
    -> 3-Fold Average Macro F1: 0.6538

--- Run 7/72 | Feature:cnn | Depth:10 | MinSamples:5 | n_feat:22 ---
    -> 3-Fold Average Macro F1: 0.6055

--- Run 8/72 | Feature:cnn | Depth:10 | MinSamples:5 | n_feat:256 ---
    -> 3-Fold 

PCA

In [ ]:
from preprocessing2 import preprocess, custom_classification_report, custom_confusion_matrix, custom_accuracy_score
X_train, y_train, X_val, y_val, X_test, y_test, weights = preprocess(feature_method="pca",n_pca=100)  # balance=True for Phase 1, False for Phase 2

import time

# 2. Initialize model
print("Initializing Custom Decision Tree...")
tree = CustomDecisionTree(
    max_depth=10,
    min_samples=10,
    n_features=X_train.shape[1],          # adjust based on feature type
)

# 3. Train model
print("Training the model...")
start_time = time.time()
tree.fit(X_train, y_train)
print(f"Training completed in {(time.time() - start_time):.2f} seconds.")

print("\n" + "="*45)
print("  CUSTOM MODEL VALIDATION PERFORMANCE")
print("="*45)

val_preds = tree.predict(X_val)

target_names = [str(i) for i in range(10)]
print(custom_classification_report(y_val, val_preds, target_names=target_names))

val_acc = custom_accuracy_score(y_val, val_preds)
print(f"Validation Accuracy: {val_acc:.4f}")

cm = custom_confusion_matrix(y_val, val_preds)

print("\nValidation Confusion Matrix (rows = actual, cols = predicted):\n")

labels = [str(i) for i in range(10)]

print(f"{'':12}", end="")
for label in labels:
    print(f"{label:>6}", end="")
print()

for i, row in enumerate(cm):
    print(f"{labels[i]:>10} ", end="")
    for val in row:
        print(f"{val:6}", end="")
    print()

print("\n" + "="*45)
print("  FINAL TEST PERFORMANCE")
print("="*45)

test_preds = tree.predict(X_test)

print(custom_classification_report(y_test, test_preds, target_names=target_names))

test_acc = custom_accuracy_score(y_test, test_preds)
print(f"Test Accuracy: {test_acc:.4f}")

cm_test = custom_confusion_matrix(y_test, test_preds)

print("\nTest Confusion Matrix (rows = actual, cols = predicted):\n")

print(f"{'':12}", end="")
for label in labels:
    print(f"{label:>6}", end="")
print()

for i, row in enumerate(cm_test):
    print(f"{labels[i]:>10} ", end="")
    for val in row:
        print(f"{val:6}", end="")
    print()

HOG

In [ ]:
from preprocessing2 import preprocess, custom_classification_report, custom_confusion_matrix, custom_accuracy_score
X_train, y_train, X_val, y_val, X_test, y_test, weights = preprocess(feature_method="hog",n_pca=50)  # balance=True for Phase 1, False for Phase 2

import time

# 2. Initialize model
print("Initializing Custom Decision Tree...")
tree = CustomDecisionTree(
    max_depth=10,
    min_samples=10,
    n_features=X_train.shape[1],          # adjust based on feature type
)

# 3. Train model
print("Training the model...")
start_time = time.time()
tree.fit(X_train, y_train)
print(f"Training completed in {(time.time() - start_time):.2f} seconds.")

val_preds = tree.predict(X_val)

target_names = [str(i) for i in range(10)]
print(custom_classification_report(y_val, val_preds, target_names=target_names))

val_acc = custom_accuracy_score(y_val, val_preds)
print(f"Validation Accuracy: {val_acc:.4f}")

cm = custom_confusion_matrix(y_val, val_preds)

print("\nValidation Confusion Matrix (rows = actual, cols = predicted):\n")

labels = [str(i) for i in range(10)]

print(f"{'':12}", end="")
for label in labels:
    print(f"{label:>6}", end="")
print()

for i, row in enumerate(cm):
    print(f"{labels[i]:>10} ", end="")
    for val in row:
        print(f"{val:6}", end="")
    print()

print("\n" + "="*45)
print("  FINAL TEST PERFORMANCE")
print("="*45)

test_preds = tree.predict(X_test)

print(custom_classification_report(y_test, test_preds, target_names=target_names))

test_acc = custom_accuracy_score(y_test, test_preds)
print(f"Test Accuracy: {test_acc:.4f}")

cm_test = custom_confusion_matrix(y_test, test_preds)

print("\nTest Confusion Matrix (rows = actual, cols = predicted):\n")

print(f"{'':12}", end="")
for label in labels:
    print(f"{label:>6}", end="")
print()

for i, row in enumerate(cm_test):
    print(f"{labels[i]:>10} ", end="")
    for val in row:
        print(f"{val:6}", end="")
    print()

Flat - with best K-folds params

In [ ]:
from preprocessing2 import preprocess, custom_classification_report, custom_confusion_matrix, custom_accuracy_score
X_train, y_train, X_val, y_val, X_test, y_test, weights = preprocess(feature_method="flatten",n_pca=50)  # balance=True for Phase 1, False for Phase 2

import time

# 2. Initialize model
print("Initializing Custom Decision Tree...")
tree = CustomDecisionTree(
    max_depth=12,
    min_samples=5,
    n_features=X_train.shape[1],          # adjust based on feature type
)

# 3. Train model
print("Training the model...")
start_time = time.time()
tree.fit(X_train, y_train)
print(f"Training completed in {(time.time() - start_time):.2f} seconds.")

val_preds = tree.predict(X_val)

target_names = [str(i) for i in range(10)]
print(custom_classification_report(y_val, val_preds, target_names=target_names))

val_acc = custom_accuracy_score(y_val, val_preds)
print(f"Validation Accuracy: {val_acc:.4f}")

cm = custom_confusion_matrix(y_val, val_preds)

print("\nValidation Confusion Matrix (rows = actual, cols = predicted):\n")

labels = [str(i) for i in range(10)]

print(f"{'':12}", end="")
for label in labels:
    print(f"{label:>6}", end="")
print()

for i, row in enumerate(cm):
    print(f"{labels[i]:>10} ", end="")
    for val in row:
        print(f"{val:6}", end="")
    print()

print("\n" + "="*45)
print("  FINAL TEST PERFORMANCE")
print("="*45)

test_preds = tree.predict(X_test)

print(custom_classification_report(y_test, test_preds, target_names=target_names))

test_acc = custom_accuracy_score(y_test, test_preds)
print(f"Test Accuracy: {test_acc:.4f}")

cm_test = custom_confusion_matrix(y_test, test_preds)

print("\nTest Confusion Matrix (rows = actual, cols = predicted):\n")

print(f"{'':12}", end="")
for label in labels:
    print(f"{label:>6}", end="")
print()

for i, row in enumerate(cm_test):
    print(f"{labels[i]:>10} ", end="")
    for val in row:
        print(f"{val:6}", end="")
    print()

Loading MNIST dataset...
Split completed: Train=54000, Val=6000, Test=10000
Initializing Custom Decision Tree...
Training the model...
Training completed in 223.38 seconds.
                 precision     recall   f1-score    support

0                     0.93       0.94       0.94        587
1                     0.94       0.97       0.95        630
2                     0.87       0.89       0.88        600
3                     0.86       0.86       0.86        627
4                     0.89       0.86       0.88        595
5                     0.84       0.83       0.84        549
6                     0.92       0.90       0.91        571
7                     0.90       0.92       0.91        668
8                     0.85       0.79       0.82        597
9                     0.82       0.85       0.83        576

accuracy                                    0.88       6000
macro avg             0.88       0.88       0.88       6000

Validation Accuracy: 0.8833

Validation Conf